# Deep Learning Training: Chest X-ray Multi-condition Classification

This notebook trains a Multi-label CNN for multi_label.

## Approach
- Transfer learning with pre-trained models
- 14-class classification
- Data augmentation for robustness
- Two-phase training strategy

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import sys
sys.path.append('../src')

# Import Kaggle downloader
try:
    from kaggle_downloader import setup_project_data, check_kaggle_credentials
    KAGGLE_AVAILABLE = True
except ImportError:
    KAGGLE_AVAILABLE = False
    print("⚠️ Kaggle downloader not available - install kaggle package")


from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Condition labels
CONDITIONS = ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema',
              'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening',
              'Cardiomegaly', 'Nodule', 'Mass', 'Hernia']

print(f'TensorFlow version: {tf.__version__}')
print(f'Number of conditions: {len(CONDITIONS)}')

## 1. Data Preparation

In [ ]:
# Load dataset
data_path = Path('../data')
print('Loading dataset...')

# Load dataset
# NOTE: Adjust file name and column names to match your dataset
csv_file = data_path / 'Data_Entry_2017.csv'

if csv_file.exists():
    df = pd.read_csv(csv_file)
    print(f"✓ Dataset loaded: {len(df)} samples")
    
    # Create one-hot encoded columns for each condition
    # Adjust 'Finding Labels' column name if your CSV uses different name
    if 'Finding Labels' in df.columns:
        for condition in CONDITIONS:
            df[condition] = df['Finding Labels'].apply(lambda x: 1.0 if condition in str(x) else 0.0)
        
        # Create image paths - adjust column and folder names if needed
        image_dir = data_path / 'images'
        if 'Image Index' in df.columns:
            df['path'] = df['Image Index'].apply(lambda x: str(image_dir / x))
        else:
            print("⚠️ 'Image Index' column not found - adjust column name above")
            df['path'] = None
        
        # Split data
        df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
        df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)
        
        print(f"✓ Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
    else:
        print("⚠️ 'Finding Labels' column not found in CSV")
        print("   Please adjust the column name in the code above")
        df_train = df_val = df_test = None
else:
    print(f"⚠️ Dataset file not found: {csv_file}")
    print("   Please:")
    print("   1. Download the NIH Chest X-ray dataset")
    print("   2. Place 'Data_Entry_2017.csv' in ../data/ folder")
    print("   3. Place images in ../data/images/ folder")
    print("   4. Update csv_file path if your file has different name")
    df_train = df_val = df_test = None

## 2. Data Generators

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(CONDITIONS)

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1/255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)

# Validation/test generator
val_test_datagen = ImageDataGenerator(rescale=1/255.0)

# Create generators
if df_train is not None:
    try:
        train_gen = train_datagen.flow_from_dataframe(
            df_train,
            x_col='path',
            y_col=CONDITIONS,
            batch_size=BATCH_SIZE,
            seed=42,
            shuffle=True,
            class_mode='raw',
            color_mode='rgb',
            target_size=IMAGE_SIZE
        )
        
        val_gen = val_test_datagen.flow_from_dataframe(
            df_val,
            x_col='path',
            y_col=CONDITIONS,
            batch_size=BATCH_SIZE,
            seed=42,
            shuffle=False,
            class_mode='raw',
            color_mode='rgb',
            target_size=IMAGE_SIZE
        )
        
        print(f"✓ Train batches: {len(train_gen)}")
        print(f"✓ Val batches: {len(val_gen)}")
    except Exception as e:
        print(f"⚠️ Error creating generators: {e}")
        print("   Check that image paths are correct and images exist")
        train_gen = val_gen = None
else:
    print("⚠️ Data not loaded - cannot create generators")
    train_gen = val_gen = None

print(f'\nConfiguration:')
print(f'  Image size: {IMAGE_SIZE}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Number of conditions: {NUM_CLASSES}')
print(f'  Conditions: {", ".join(CONDITIONS[:5])}...')

## 3. Model Architecture

In [ ]:
from model import build_chest_xray_model

# Build model
model = build_chest_xray_model(input_shape=(*IMAGE_SIZE, 3), num_classes=NUM_CLASSES)

print('Model architecture:')
print(f'Total parameters: {model.count_params():,}')
model.summary()

## 4. Training

In [ ]:
# Model is already compiled, but we can adjust
# For multi-label, we use binary_crossentropy with sigmoid activation

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint('../models/chest_xray_model.h5', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

# Train
EPOCHS = 50

# Train model
if train_gen is not None and val_gen is not None:
    print("Starting training...")
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )
    print("\n✓ Training complete!")
else:
    print("⚠️ Cannot train - data generators not created")
    print("   Please load data and create generators in cells above")
    history = None

print(f'\nConfiguration:')
print(f'  Epochs: {EPOCHS}')
print('  Note: Multi-label classification - each condition predicted independently')

## 5. Evaluation

In [ ]:
# Evaluate model
if df_test is not None and model is not None:
    try:
        test_gen = val_test_datagen.flow_from_dataframe(
            df_test,
            x_col='path',
            y_col=CONDITIONS,
            batch_size=BATCH_SIZE,
            seed=42,
            shuffle=False,
            class_mode='raw',
            color_mode='rgb',
            target_size=IMAGE_SIZE
        )
        
        test_results = model.evaluate(test_gen, verbose=1)
        print(f'\n✓ Test Loss: {test_results[0]:.4f}')
        print(f'✓ Test Accuracy: {test_results[1]:.4f}')
        
        # Make predictions
        test_gen.reset()
        predictions = model.predict(test_gen, verbose=0)
        y_pred = (predictions > 0.5).astype(int)  # Threshold for multi-label
        y_true = df_test[CONDITIONS].values[:len(y_pred)]
        
        # Per-condition metrics
        print('\n' + '='*60)
        print('Per-Condition Performance:')
        print('='*60)
        from sklearn.metrics import precision_score, recall_score, f1_score
        for i, condition in enumerate(CONDITIONS):
            condition_pred = y_pred[:, i]
            condition_true = y_true[:, i]
            precision = precision_score(condition_true, condition_pred, zero_division=0)
            recall = recall_score(condition_true, condition_pred, zero_division=0)
            f1 = f1_score(condition_true, condition_pred, zero_division=0)
            print(f'{condition:20s}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}')
    except Exception as e:
        print(f"⚠️ Error during evaluation: {e}")
else:
    print("⚠️ Cannot evaluate - data or model not available")
    print("   Please load data and train model first")